# Runtime
* run `./ableton_projects/start_notebook.sh` to start
server on `8888`
* see `start_notebook.sh` for installing python2 kernel and packages
* see https://research.google.com/colaboratory/local-runtimes.html


# Modules

In [1]:
#@title Imports
import pandas as pd
import matplotlib.pyplot as plt


from pprint import pprint
import os
import pickle
import time
from datetime import datetime

import sys
print('Python Version: %s' % sys.version)

Python Version: 3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]


In [ ]:
#@title Library

import xml.etree.ElementTree as ET
import gzip
import plistlib
# the maybe monad
def bind(val, f):
    return (val is not None) and f(val) or None

hexchars = set('0123456789abcdef')
def decodeHexString(str):
    """Decode a hex string, ignoring all non-hex characters"""
    return filter(lambda c:c in hexchars, str.lower()).decode('hex')

def hideUnprintable(str, maskchar='.'):
    return ''.join(map(lambda c:(ord(c)>0x1f and ord(c)<0x7e) and c or maskchar, str))

def decodeTimeSignature(encoded):
    """Convert an Ableton Live integer-encoded time signature into a
       (numerator,denominator) tuple"""
    return (encoded%99+1, 1<<(encoded/99))

#  ---- classes

# the boolean value type:
def BoolValue(str): return  (str == "true")

# return the type function for a string value
def guessTypeForValue(v):
    if (v in ('true','false')): return BoolValue
    for fn in (int, float):
        try:
            if fn(v) is not None:
                return fn
        except ValueError:
            pass
    return None  # just a string

class ALSNode(object):
    """
    The parent class of all .als nodes
    """

    # the value fields we automatically extract from the element; each one
    # has a name mapped to a type/conversion function; i.e., "Time" : float .

    valuefields = {}

    def __init__(self, elem):
        self.elem = elem
        # populate instance vars using harvested Value attributes from the element or subelements
        for (key, fspec) in self.valuefields.iteritems():
            if type(fspec) == dict:
                (sel, ivar, vtype ) = (fspec.get('sel',key),fspec.get('ivar',key),fspec.get('type',None))
            elif type(fspec) == tuple:  # A simple (type, sel) tuple
                (sel, ivar, vtype) = (fspec[1], key, fspec[0])
            else:
                (sel, ivar, vtype) = (key, key[0].lower()+key[1:], fspec)
            val = self.valueForSubtag(sel)
            if val:
                if vtype:
                    try:
                        val = vtype(val)
                    except ValueError:
                        pass
            self.__dict__[ivar] = val


    def valueForSubtag(self, selector):
        return bind(self.elem.find(selector), lambda e: e.get("Value"))

    def valueForSubtagWithType(self, selector, type):
        if type is None: type = lambda x:x
        try:
            return type(self.valueForSubtag(selector))
        except ValueError:
            return None

    def intValueForSubtag(self, selector):
        return self.valueForSubtagWithType(selector, int)

    def floatValueForSubtag(self, selector):
        return self.valueForSubtagWithType(selector, float)

    def boolValueForSubtag(self, selector):
        return self.valueForSubtag(selector) == 'true'


# A node representing an automatable parameter

class ALSTrackMixerParam(ALSNode):
    def __init__(self, elem):
        super(ALSTrackMixerParam, self).__init__(elem)
        # determine the element type
        manual = self.valueForSubtag("Manual")
        self.type = guessTypeForValue(manual)
        typefunc = self.type and self.type or (lambda x:x)
        self.manual = typefunc(manual)
        self.events = [ (int(e.get('Time')), typefunc(e.get('Value'))) for e in elem.findall("ArrangerAutomation/Events/*")]


class ALSTrackMixer(ALSNode):
    def __init__(self, elem):
        super(ALSTrackMixer, self).__init__(elem)
        self.params = dict([(e.tag, ALSTrackMixerParam(e)) for e in elem.findall("*[ArrangerAutomation]")])

# Clips and their component classes

class ALSWarpMarker(object):
    def __init__(self, elem):
        self.secTime = float(elem.get('SecTime'))
        self.beatTime = float(elem.get('BeatTime'))

class ALSMidiNote(object):
    def __init__(self, key, elem):
        # for some reason, Live stores MIDI velocities as floating-point values
        self.time, self.key, self.duration, self.velocity, self.offVelocity, self.isEnabled = (float(elem.get("Time")), key, float(elem.get("Duration")), float(elem.get("Velocity")), int(elem.get("OffVelocity")),
            elem.get("IsEnabled")=="true")

class LiveSetMidiClipData(ALSNode):
    """
    An object encapsulating a MidiClip node.
    """
    valuefields = {
      'Name': None, 'Annotation': None, 'LaunchMode':int, 'CurrentStart':float, 'CurrentEnd':float,
      'loopStart' : (float, "Loop/LoopStart"), 'loopEnd' : (float, "Loop/LoopEnd"),
      'loopStartRelative' : (float, "Loop/LoopStartRelative"),
    }
    def __init__(self, elem):
        super(LiveSetMidiClipData, self).__init__(elem)
        self.warpmarkers = [ ALSWarpMarker(e) for e in elem.findall("WarpMarkers/WarpMarker")]
        self.length = (self.currentStart is not None and self.currentEnd is not None) and self.currentEnd-self.currentStart or None
        self.loopOn = self.boolValueForSubtag("Loop/LoopOn")
        self.loopLength = (self.loopStart is not None and self.loopEnd is not None) and self.loopEnd-self.loopStart or None

        self.notes = []
        for ktrk in elem.findall("Notes/KeyTracks/KeyTrack"):
            note = bind(ktrk.find("MidiKey"), lambda e:int(e.get("Value")))
            self.notes.extend([ALSMidiNote(note, mne) for mne in ktrk.findall("Notes/MidiNoteEvent")])
        self.notes.sort(key=lambda mn:mn.time)


# Devices

class LiveSetAuPluginPresetData(object):
    """
    An object encapsulating the data stored in a preset buffer
    """
    def __init__(self, text):
        self.text = decodeHexString(text)
        self.plist = plistlib.readPlistFromString(self.text)
        self.name = self.plist.get('name')

class LiveSetDeviceData(ALSNode):
    """
    An object encapsulating the data for a device
    """
    def __init__(self, elem):
        super(LiveSetDeviceData, self).__init__(elem)
        self.deviceType = elem.tag
        self.auPresetBuffer = bind(elem.find("PluginDesc/AuPluginInfo/Preset/AuPreset/Buffer"), lambda e:LiveSetAuPluginPresetData(e.text))
        self.auPresetName = bind(self.auPresetBuffer, lambda b:b.name)

        self.presetName = self.valueForSubtag("UserName") \
                or bind(elem.find("PluginDesc/AuPluginInfo/Name"), lambda e:e.get("Value")) \
                or self.valueForSubtag("PluginDesc/VstPluginInfo/PlugName") \
                or ""
        self.name = "%s: %s"%(self.deviceType, self.presetName)

# Tracks

class LiveSetTrackData(ALSNode):
    """
    An object encapsulating the data for a Track
    """
    valuefields = { 'Name' : None }
    def __init__(self, elem):
        super(LiveSetTrackData, self).__init__(elem)
        self.trackType = elem.tag
        self.devices = [LiveSetDeviceData(c) for c in elem.find("DeviceChain/DeviceChain/Devices")]

        self.mixer = bind(elem.find("DeviceChain/Mixer"), ALSTrackMixer)

        # TODO: encapsulate these in a class
        self.clipslots = bind(elem.find("DeviceChain/MainSequencer/ClipSlotList"), lambda x:x.findall("ClipSlot")) or []
        self.midiclips = bind(elem.find("DeviceChain/MainSequencer/ClipSlotList"), lambda x:[LiveSetMidiClipData(c) for c in x.findall(".//MidiClip")]) or []


class LiveSetData(object):
    """
    An object encapsulating a parsed Live set.
    """
    def __init__(self, path):
        self.etree = ET.parse(gzip.GzipFile(path))
        self.live_set = self.etree.getroot().find("LiveSet")
        self.tracks = [LiveSetTrackData(c) for c in self.live_set.find("Tracks")]
        self.mastertrack = bind(self.live_set.find("MasterTrack"), LiveSetTrackData)

    def timeSignatures(self):
        """Return an array of (beat time, (num,denom)) time signatures used in the track."""
        return [(max(t,0), decodeTimeSignature(enc)) for (t,enc) in self.mastertrack.mixer.params['TimeSignature'].events]


In [ ]:
#@title Parse .als

def parse_als_info(path):
    """Print out some info about an Ableton Live set at a path"""
    lsd = LiveSetData(path)
    name, ext = os.path.splitext(path)
    assert ext == '.als'
    # pprint(lsd.__dict__)
    info = {
        'path': path,
        'name': os.path.basename(name),
        'project': os.path.dirname(name).split('/')[-1].replace(' Project', ''),
        'tracks': []
    }
    all_tracks = lsd.tracks + [lsd.mastertrack]
    for i, t in enumerate(all_tracks):
      i = i+1 # start at 1
      # pprint(t.__dict__)
      # Note: clipSLots field not used (not sure what its good for).
      #   Also not using .elem (which is a raw field)

      track = {'index': i, 'type': t.trackType, 'devices': [], 'clips': []}
      for dev in t.devices:
        track['devices'].append({
            'type': dev.deviceType,
            'preset': dev.presetName
        })

      for clip in t.midiclips:
        track['clips'].append({
            'name': clip.name,
            'length': clip.length,
            'is_loop': clip.loopOn
        })

      info['tracks'].append(track)

      # FIXME: seems these fields are unset or broken
      if t.mixer.params != {}:
        print(' DEBUG: track has mixer params most likely older project')
      if t.name != None:
        print(' DEBUG: track has name(which is rare): %s' % t.name)

    return info


# Testing
# project_dir = './'
# project_file = project_dir + '__Dimensions_/1 whirl tron Project/1 headphones.als'
# project_file = project_dir + '_guitar beats/raw Project/raw.als'
# parse_als_info(project_file)

In [2]:
#@title Get plugins from all projects

def query_projects_by_plugin(project_info, plugin_type, preset_name):
  project_keys = []
  assert plugin_type in ['PluginDevice', 'AuPluginDevice']
  for fname, v in project_info.items():
    for track in v['tracks']:
      for dev in track['devices']:
        if dev['type'] == plugin_type and dev['preset'] == preset_name:
            project_keys.append('%s-%d' % (fname, track['index']))

  return project_keys

def get_plugin_usage(project_info):
  plugins = {'vst': [], 'audio_unit': [], 'ableton': []}
  for fname, v in project_info.items():
    for track in v['tracks']:
      for dev in track['devices']:
        if dev['type'] == 'PluginDevice': # for vsts
          plugins['vst'].append(dev['preset'])
        elif dev['type'] == 'AuPluginDevice':  # for audiounits
          plugins['audio_unit'].append(dev['preset'])
        else:  # for default ableton plugin
          plugins['ableton'].append(dev['type'])
  plugins_usage = {k:pd.value_counts(pd.Series(v)) for k,v in plugins.items()}
  return plugins_usage

# # TEST
# plugins = get_plugin_usage(project_info)
# pprint(plugins)

# query_projects_by_plugin(project_info,
#     plugin_type='PluginDevice',
#     preset_name='Maschine 2'
# )

In [ ]:
#@title Save and Load Info
def save_info(cache_dir, info_dict, prefix):
    save_path = os.path.join(cache_dir, '%s.%d' % (prefix, time.time()))
    timestamp = int(save_path.split('.')[-1])
    save_date = datetime.fromtimestamp(timestamp)
    print('Saving %s from %s.' % (prefix, save_date))
    with open(save_path, 'wb') as handle:
        pickle.dump(info_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)
    return save_path

def load_info(cache_dir, prefix='project_info', cache_file='', load_most_recent=True):
  cache = sorted([f for f in os.listdir(cache_dir) if prefix in f])
  if cache_file not in cache or load_most_recent:
    cache_file = cache[-1]
  cache_path = os.path.join(cache_dir, cache_file)
  timestamp = int(cache_path.split('.')[-1])
  cache_date = datetime.fromtimestamp(timestamp)
  t0 = time.time()
  print('Loading %s from %s.' % (prefix, cache_date))
  with open(cache_path, 'rb') as handle:
    info_dict = pickle.load(handle)
  print('Loaded %d projects in %d seconds.'  % (
      len(info_dict), time.time()-t0))
  return info_dict


def load_projects_in_dir(project_dir, skip_folders):
  project_info = {}
  project_errors = {}
  t0 = time.time()
  print('Loading projects in %s.' % project_dir)
  for (dirpath, dirnames, filenames) in os.walk(project_dir):
    if any(f in dirpath for f in skip_folders):
      # print 'skipping ' + dirpath
      continue

    for filename in filenames:
      key, ext = os.path.splitext(filename)
      if ext == '.als' and not key.startswith('.') and not '~' in key:
        full_filename = os.sep.join([dirpath, filename])
        print(' READING: %s' % key)
        try:
          project_info[key] = parse_als_info(full_filename)
        except Exception as e:
          project_errors[key] = e
          print(' ERROR: %s' % e)
  print('Loaded %d projects with %d error projects in %d seconds.' % (
      len(project_info), len(project_errors), time.time()-t0))
  return project_info, project_errors



#
## TEST

# # Load info (slow)
# project_dir = "./" #@param {type:"string"}
# skip_folders = ['source projects', 'Backup', 'old', 'Samples', 'Ableton Project Info'] #@param {type:"raw"}
# project_info, project_errors = load_projects_in_dir(project_dir, skip_folders)

# prefix = 'project_info'
# saved_f = save_info(cache_dir, project_info, prefix)

# # load specific
# res = load_info(cache_dir, cache_file='project_info.1572831091', load_most_recent=False)

# # load newest
# res = load_info(cache_dir, load_most_recent=True)

# Analysis

In [5]:
#@title Get info for all Projects and cache

project_dir = "./" #@param {type:"string"}
skip_folders = [ 'Backup', 'old', 'Samples', 'Ableton Project Info', '.git'] #@param {type:"raw"}

cache_dir = "./logs/" #@param {type:"string"}
cache_prefix = "project_info" #@param {type:"string"}
cache_info = True #@param {type:"boolean"}
load_most_recent_cached = True #@param {type:"boolean"}

# Load project info
if load_most_recent_cached: # Load from recent cache (fast)
  project_errors = {}
  project_info = load_info(cache_dir, prefix=cache_prefix, load_most_recent=True)
  cache_info = False
else: # Parse .als files
  project_info, project_errors = load_projects_in_dir(project_dir, skip_folders)

# Cache info
if cache_info:
  if project_info != {}:
    save_info(cache_dir, project_info, prefix=cache_prefix)
  if project_errors != {}:
    save_info(cache_dir, project_info, prefix=cache_prefix+'_errors')


Loading project_info from 2024-11-28 00:46:25.
Loaded 6 projects in 0 seconds.


In [6]:
for k in project_info.keys():
  if 'lekato' in k:
    print(k)

In [ ]:
#@title Show VST and AU usage

# Show histogram of plugins
skip_keys = ['ableton']
colors = ['green', 'red', 'blue']
n=1000
plugins = get_plugin_usage(project_info)
for key in plugins.keys():
  if key in skip_keys:
    continue
  print('%d least used %s plugins:\n%s' % (n, key, plugins[key][-n:]))
  plugins[key].plot.barh(figsize=(23, 12), grid=True, color=colors.pop(), title=key)
  plt.show()

In [ ]:
query_projects_by_plugin(project_info, "PluginDevice", "Neutron 4")

In [ ]:
#@title Some checks
DEPRECATED_PLUGINS = ["Ozone 8", "Ozone 6"] # "Neutron 3"
for vst in DEPRECATED_PLUGINS:
  print(f'CHECK\t| {vst} should not be an any projects', end='')
  assert 0 == len(query_projects_by_plugin(project_info, "PluginDevice", vst))
  print('...and is not')

ONLY_A10_AUDIO_UNIT = ['Crystallizer', 'GS-201']
for au in ONLY_A10_AUDIO_UNIT:
  print(f'NOTE\t| {au} is only available as a 32 bit audio unit')


CHECK	| Ozone 8 should not be an any projects...and is not
CHECK	| Ozone 6 should not be an any projects...and is not
CHECK	| Neutron 3 should not be an any projects

AssertionError: 

#### TODO: FIX Audiounit plugins
* dimensions 4: on a10, swap out iZotope Ozone 5 AU for VST
* slow burn mixdown: swap out Maschine 2 and Komplete Kontrol AU for VST
* brady-jake jam +extra day 2: swap out Massive AU for VST
* brady-jake jam 0: swap out Massive AU for VST
* acoustic sad vox (multiple): swap out iZotope Nectar 2 AU for VST

**TODO**:add above check that the only 2 audiounites are ONLY_A10_AUDIO_UNIT


In [ ]:
#@title Query
plugin_type = "PluginDevice" #@param ["PluginDevice", "AuPluginDevice"]
preset_name = "Reaktor 6 FX" #@param {type:"string"}

projects = query_projects_by_plugin(project_info,
    plugin_type=plugin_type,
    preset_name=preset_name
)

print('Projects matching preset: %s\n' % preset_name)
for p in sorted(projects):
  print(p)

Projects matching preset: Reaktor 6 FX

2021 mac13 template-1
2021 mac13 template-2
MORE of a jam-10
NEW YTEMPLATE-2
Untitled-1
Untitled-2
a-1
a-2
aciustic mac feedback-1
aciustic mac feedback-2
acoustic jam psych-1
acoustrunkv2-2
acoustrunkv2-3
acoustrunkv2-4
acoustrunkv2-5
acoustrunkvo-1
acoustrunkvo-2
autotune sad minor guitar-3
bnchmark-1
bnchmark-2
current template-2
death flutes-6
garage rock template-3
guitar live-1
guitar vox loops downmixed-1
guitar vox loops downmixed-2
guitar vox loops-1
guitar vox loops-2
hipohopy-1
hipohopy-2
loose acousitc distorted vox loop jam-1
makings of a jam-2
new roume jaamz-1
psychccy acoust trip loops-3
sad verse psych-1
sad verse psych-2
single trackpyshch new roume accoustic blah-1
slow dark garage jaaam-3
soul sample-3
template_2020 2-1
template_2020 2-2
template_2020-1
template_2020-2
traoppy-6


In [ ]:
#@title Audio Unit Query
plugin_type = "AuPluginDevice" #@param ["PluginDevice", "AuPluginDevice"]
preset_name = "GS-201" #@param {type:"string"}

projects = query_projects_by_plugin(project_info,
    plugin_type=plugin_type,
    preset_name=preset_name
)

print('Projects matching preset: %s\n' % preset_name)
for p in sorted(projects):
  print(p)

Projects matching preset: GS-201

1 2022-27
1-27
2-14
3-7
4-22
461-6
5-14
5-15
Blazed Electric Guitar recorded-8
Dance Beat [Loops]-7
Dowm Chill 'vErB up-14
Downtime Chill Vibe-6
Green the color-8
Maggots Chill-5
SpokoACIDsampler [song] [unmxd]-10
b itch gowsays [Loops]-3
b itch gowsays [Loops]-7
dimension live remix 2-7
dimension live remix 2-9
hazzy chill-7
high vox loops-2
high vox loops-3
high vox loops-5
high vox loops-7
jarryork-5
loaunge chill j a10 gsi-8
lofi accoustic psych-7
new bumpabumb-15
psych guiTarr beat-2
psych guiTarr beat-4
something-7
surff classicaL sh-3
trippy vox loops-2
trippy vox loops-3
trippy vox loops-5
trippy vox loops-7
watching fish-11
